# Day 3 — Reranking, MMR & Metadata Filtering

Two-stage retrieval: Stage 1 (BM25/dense) retrieves 50 fast candidates.
Stage 2 (cross-encoder) reranks the 50 with higher accuracy to get top 5.
This is industry standard at Google, Microsoft, and every serious search system.

In [ ]:
import sys
sys.path.insert(0, '../src')
import numpy as np
from day3.reranking import (
    CrossEncoderReranker, mmr_select,
    metadata_filter, build_retrieval_pipeline, RankedResult
)
from day3.hybrid_search import DenseIndex, generate_product_corpus

def mock_embed(texts):
    vecs = []
    for t in texts:
        rng = np.random.default_rng(abs(hash(t)) % (2**31))
        v = rng.standard_normal(384).astype(np.float32)
        v = v / np.linalg.norm(v)
        vecs.append(v)
    return np.array(vecs)

corpus = generate_product_corpus()
print(f"Corpus: {len(corpus)} product documents loaded")

## 1. Cross-Encoder Reranking

Bi-encoders (dense retrieval) embed query and document SEPARATELY — fast but approximate.
Cross-encoders process (query, document) TOGETHER through full transformer attention — 10x slower but much more accurate.
We use mock reranking (Jaccard word overlap) to avoid model downloads.

In [ ]:
reranker = CrossEncoderReranker()
query = "noise cancelling wireless headphones for travel"
candidates = corpus[:10]  # simulate 10 first-stage results

reranked = reranker.rerank_mock(query, candidates, top_k=5)
print(f"Query: '{query}'")
print(f"\nReranked top-5:")
for r in reranked:
    change = "\u2191" if r.final_rank < r.initial_rank else ("\u2193" if r.final_rank > r.initial_rank else "=")
    print(f"  [{r.final_rank}] {change} (was #{r.initial_rank}) "
          f"score={r.final_score:.4f} | {r.text[:80]}")
    
changed = sum(1 for r in reranked if r.rank_changed)
print(f"\n{changed}/{len(reranked)} documents changed position after reranking")

## 2. Real Cross-Encoder (Optional)

With ANTHROPIC_API_KEY or a downloaded model, use the real cross-encoder.
In production, cross-encoder/ms-marco-MiniLM-L-6-v2 is a fast, high-quality choice for passage reranking.

In [ ]:
try:
    # Real cross-encoder \u2014 only runs if model downloads are allowed
    real_reranker = CrossEncoderReranker("cross-encoder/ms-marco-MiniLM-L-6-v2")
    real_reranked = real_reranker.rerank(query, candidates[:5], top_k=3)
    print("Real cross-encoder results:")
    for r in real_reranked:
        print(f"  [{r.final_rank}] score={r.final_score:.4f} | {r.text[:80]}")
except Exception as e:
    print(f"Real cross-encoder skipped: {e}")
    print("Using mock reranker for offline demonstration")

## 3. Maximal Marginal Relevance (MMR)

MMR balances relevance to the query against diversity among selected docs.
Formula: \u03bb * sim(query, doc) - (1-\u03bb) * max(sim(selected, doc)).
Lambda=1.0 = pure relevance, Lambda=0.0 = pure diversity, Lambda=0.5 = balanced (recommended).

In [ ]:
dense = DenseIndex(mock_embed_fn=mock_embed)
dense.index(corpus)

query_laptop = "best laptop for software development"
query_vec = dense._encode([query_laptop])[0]
doc_vecs = dense._embeddings

print(f"Query: '{query_laptop}'")
print("\nlambda=1.0 (pure relevance \u2014 may return duplicates):")
for idx, score in mmr_select(query_vec, doc_vecs, corpus, top_k=4, lambda_param=1.0):
    print(f"  [{idx}] score={score:.4f} | {corpus[idx][:80]}")
    
print("\nlambda=0.5 (balanced relevance + diversity):")
for idx, score in mmr_select(query_vec, doc_vecs, corpus, top_k=4, lambda_param=0.5):
    print(f"  [{idx}] score={score:.4f} | {corpus[idx][:80]}")
    
print("\nlambda=0.0 (maximum diversity):")
for idx, score in mmr_select(query_vec, doc_vecs, corpus, top_k=4, lambda_param=0.0):
    print(f"  [{idx}] score={score:.4f} | {corpus[idx][:80]}")

## 4. Metadata Filtering

Pre-filter documents by category, price range, or tags BEFORE running vector search.
This narrows the search space (e.g., 200 laptop docs instead of 10,000 total), improving both speed and precision.

In [ ]:
# Build documents with metadata
docs_with_meta = []
categories = {"laptop": ["macbook", "thinkpad", "xps", "zephyrus", "surface pro"],
               "headphones": ["wh-1000", "airpods", "bose", "jabra"],
               "smartphone": ["galaxy s24", "iphone", "oneplus", "pixel", "xiaomi"],
               "tablet": ["ipad", "galaxy tab", "kindle"],
               "accessories": ["logitech", "anker", "samsung t7"]}

for text in corpus:
    text_lower = text.lower()
    cat = "other"
    for category, keywords in categories.items():
        if any(kw in text_lower for kw in keywords):
            cat = category
            break
    price_val = 150000 if "\u20b91," in text else 50000
    docs_with_meta.append({
        "text": text,
        "metadata": {"category": cat, "price": price_val}
    })

# Filter by category
laptop_docs = metadata_filter(docs_with_meta, {"category": "laptop"})
print(f"Total: {len(docs_with_meta)} | Laptop filter: {len(laptop_docs)} docs")

headphone_docs = metadata_filter(docs_with_meta, {"category": "headphones"})
print(f"Headphones filter: {len(headphone_docs)} docs")

# Filter by price range
budget_docs = metadata_filter(docs_with_meta, {"price_range": (0, 60000)})
print(f"Budget (\u20b90\u201360K) filter: {len(budget_docs)} docs")

## 5. Full Retrieval Pipeline

Chain metadata filter \u2192 dense retrieval \u2192 reranking into a single callable.
This is the production pattern for a RAG system that needs to respect category/price constraints while still ranking by semantic relevance.

In [ ]:
retrieve = build_retrieval_pipeline(docs_with_meta, [], mock_embed_fn=mock_embed)

print("Full pipeline: filter \u2192 dense \u2192 rerank\n")
test_queries_meta = [
    ("best laptop for productivity",    {"category": "laptop"}),
    ("noise cancelling audio",          {"category": "headphones"}),
    ("affordable portable device",      {}),
]

for q, filters in test_queries_meta:
    results = retrieve(q, filters=filters, top_k=3)
    print(f"Query: '{q}'  filters={filters or 'none'}")
    for r in results:
        print(f"  [{r.final_rank}] score={r.final_score:.4f} | {r.text[:80]}")
    print()

## Databricks Bridge

In Databricks, metadata filtering is passed directly to the vector search query.
No separate filter step needed.

In [ ]:
print("""
DATABRICKS VECTOR SEARCH \u2014 METADATA FILTER:

  results = index.similarity_search(
      query_text = "noise cancelling wireless headphones",
      columns    = ["chunk_id", "text", "category", "price"],
      num_results = 10,
      filters    = {"category": "headphones", "price <": 35000},
  )
  
  # Then rerank with Mosaic ML Reranking API:
  from databricks.reranking import Reranker
  reranker = Reranker(endpoint_name="databricks-bge-reranker-v2-m3")
  reranked = reranker.rerank(
      query       = query,
      passages    = [r["text"] for r in results],
      top_n       = 5,
  )
""")